# 04 — Silver: Forecast 5 Dias
## Módulo 2B | Camada Silver | Seção do Exame: S3 (22%)

**O que este notebook faz:**
- Lê `bronze.raw_forecast` (payload JSON com array `list[]` de 40 previsões)
- **Explode o array** `list[]` → uma linha por slot de 3h por cidade
- Extrai e tipifica cada campo do slot de previsão
- Aplica Data Quality e faz `MERGE INTO` na Silver

**Conceito-chave:** `explode()` é a função PySpark que transforma um array  
em múltiplas linhas — fundamental para dados aninhados como este.

---

### ✅ Checklist deste notebook
- [ ] Célula 1 — Schema do forecast definido
- [ ] Célula 2 — `from_json` + `explode`: `df.count()` retorna ~160 linhas (40 × 4 cidades)
- [ ] Célula 3 — Campos extraídos com tipos corretos
- [ ] Célula 4 — DQ aplicado
- [ ] Célula 5 — Tabela `silver.forecast` criada
- [ ] Célula 6 — MERGE INTO executado
- [ ] Re-executei e count Silver não mudou (idempotência)

---
## Célula 1 — Schema do Forecast JSON

**Estrutura do JSON `/forecast`:**
```json
{
  "city": {"id": 3448439, "name": "São Paulo", "country": "BR", "coord": {...}},
  "list": [
    {
      "dt": 1704067200, "dt_txt": "2024-01-01 00:00:00",
      "main": {"temp": 22.5, "temp_min": 20.1, "temp_max": 24.3},
      "weather": [{"main": "Rain", "description": "light rain"}],
      "pop": 0.85
    },
    ... (39 slots mais)
  ]
}
```

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, ArrayType,
    IntegerType, LongType, DoubleType, FloatType, StringType
)
from delta.tables import DeltaTable

dbutils.widgets.text("catalog", "workshop_weather", "Catálogo UC")
CATALOG      = dbutils.widgets.get("catalog")
BRONZE_TABLE = f"{CATALOG}.bronze.raw_forecast"
SILVER_TABLE = f"{CATALOG}.silver.forecast"

# Schema de cada item dentro do array list[]
SLOT_SCHEMA = StructType([
    StructField("dt",     LongType(),   True),
    StructField("dt_txt", StringType(), True),
    StructField("main", StructType([
        StructField("temp",     FloatType(), True),
        StructField("temp_min", FloatType(), True),
        StructField("temp_max", FloatType(), True),
        StructField("humidity", IntegerType(), True),
    ]), True),
    StructField("weather", ArrayType(StructType([
        StructField("main",        StringType(), True),
        StructField("description", StringType(), True),
    ])), True),
    StructField("pop", DoubleType(), True),  # probability of precipitation [0.0, 1.0]
])

# Schema do objeto city dentro do payload
CITY_SCHEMA = StructType([
    StructField("id",      IntegerType(), True),
    StructField("name",    StringType(),  True),
    StructField("country", StringType(),  True),
    StructField("coord", StructType([
        StructField("lat", DoubleType(), True),
        StructField("lon", DoubleType(), True),
    ]), True),
])

# Schema completo do payload
FORECAST_SCHEMA = StructType([
    StructField("city", CITY_SCHEMA,             True),
    StructField("list", ArrayType(SLOT_SCHEMA),  True),
])

print("✅ Schema definido.")

---
## Célula 2 — Parsing + `explode()` do array `list[]`

**O que `explode()` faz:**
```
Antes:  1 linha por cidade, com array de 40 slots
Depois: 40 linhas por cidade — uma por slot de 3h

4 cidades × 40 slots = 160 linhas
```

In [ ]:
df_bronze = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("_status") == "success")
    .filter(F.col("raw_payload").isNotNull())
)

# Parseia o JSON completo
df_parsed = df_bronze.withColumn(
    "data",
    F.from_json(F.col("raw_payload"), FORECAST_SCHEMA)
)

# explode() transforma o array list[] em uma linha por slot
df_exploded = df_parsed.select(
    F.col("data.city")                        .alias("city"),
    F.explode(F.col("data.list"))             .alias("slot"),
    F.col("_batch_id"),
    F.col("_ingestion_timestamp"),
)

total = df_exploded.count()
print(f"Linhas após explode: {total}")
print(f"Esperado           : ~{df_bronze.count() * 40} (cidades × 40 slots)")
print()
df_exploded.select(
    F.col("city.name"), F.col("slot.dt_txt"), F.col("slot.main.temp")
).show(5, truncate=False)

---
## Célula 3 — Extração e Tipagem dos Campos

In [ ]:
df_staging = df_exploded.select(
    # Identificação da cidade
    F.col("city.id")                            .alias("city_id"),
    F.col("city.name")                          .alias("city_name"),
    F.col("city.country")                       .alias("country"),
    F.col("city.coord.lat")                     .alias("lat"),
    F.col("city.coord.lon")                     .alias("lon"),
    # Timestamp da previsão
    F.from_unixtime(F.col("slot.dt")).cast("timestamp").alias("forecast_timestamp"),
    F.col("slot.dt_txt")                        .alias("forecast_dt_txt"),
    # Temperatura
    F.col("slot.main.temp")                     .alias("temp"),
    F.col("slot.main.temp_min")                 .alias("temp_min"),
    F.col("slot.main.temp_max")                 .alias("temp_max"),
    F.col("slot.main.humidity")                 .alias("humidity"),
    # Condição
    F.col("slot.weather")[0]["main"]            .alias("weather_main"),
    F.col("slot.weather")[0]["description"]     .alias("weather_description"),
    # Probabilidade de precipitação [0.0, 1.0]
    F.col("slot.pop")                           .alias("pop"),
    # Metadados
    F.col("_batch_id"),
    F.col("_ingestion_timestamp"),
    F.current_timestamp()                       .alias("_processed_timestamp"),
)

print("Amostra após extração:")
df_staging.select(
    "city_name", "forecast_timestamp", "temp", "weather_main", "pop"
).show(8, truncate=False)

---
## Célula 4 — Data Quality

In [ ]:
df_with_dq = df_staging.withColumn(
    "_dq_status",
    F.when(
        F.col("city_id").isNull() |
        F.col("forecast_timestamp").isNull() |
        F.col("temp").isNull() |
        (F.col("temp") < -80) | (F.col("temp") > 60) |
        (F.col("pop").isNotNull() & ((F.col("pop") < 0) | (F.col("pop") > 1))),
        F.lit("invalid")
    ).otherwise(F.lit("valid"))
)

df_with_dq.groupBy("_dq_status").count().show()
print("✅ DQ aplicado.")

---
## Célula 5 — Criar a Tabela Silver Forecast

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS workshop_weather.silver.forecast (
    city_id              INT        COMMENT 'ID da cidade na OpenWeatherMap',
    city_name            STRING     COMMENT 'Nome da cidade',
    country              STRING     COMMENT 'Código do país',
    lat                  DOUBLE,
    lon                  DOUBLE,
    forecast_timestamp   TIMESTAMP  COMMENT 'Timestamp da previsão (UTC)',
    forecast_dt_txt      STRING     COMMENT 'Data/hora da previsão em texto (API)',
    temp                 FLOAT      COMMENT 'Temperatura prevista (°C)',
    temp_min             FLOAT      COMMENT 'Temperatura mínima prevista (°C)',
    temp_max             FLOAT      COMMENT 'Temperatura máxima prevista (°C)',
    humidity             INT        COMMENT 'Umidade relativa (%)',
    weather_main         STRING     COMMENT 'Condição principal da previsão',
    weather_description  STRING,
    pop                  DOUBLE     COMMENT 'Probabilidade de precipitação [0.0, 1.0]',
    _batch_id            STRING,
    _ingestion_timestamp TIMESTAMP,
    _processed_timestamp TIMESTAMP,
    _dq_status           STRING
)
USING DELTA
COMMENT 'Tabela Silver: previsão 5 dias por cidade, explodida por slot de 3h'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

---
## Célula 6 — MERGE INTO

**Chave de negócio:** `city_id + forecast_timestamp`  
Uma cidade tem uma única previsão por slot de 3h.

In [ ]:
df_with_dq.createOrReplaceTempView("staging_forecast")

spark.sql(f"""
MERGE INTO {SILVER_TABLE} AS target
USING staging_forecast AS source
ON  target.city_id           = source.city_id
AND target.forecast_timestamp = source.forecast_timestamp
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

print("✅ MERGE INTO executado.")

---
## Célula 7 — Validação

In [ ]:
%sql
SELECT
    city_name,
    COUNT(*)                   AS slots,
    MIN(forecast_timestamp)    AS primeiro_slot,
    MAX(forecast_timestamp)    AS ultimo_slot,
    ROUND(AVG(temp), 1)        AS avg_temp,
    ROUND(MAX(pop) * 100, 0)   AS max_rain_prob_pct
FROM workshop_weather.silver.forecast
GROUP BY city_name
ORDER BY city_name;

---
## ✅ Checklist Final — Módulo 2B concluído?

- [ ] Count após explode é ~160 (40 slots × 4 cidades)
- [ ] `forecast_timestamp` é do tipo `timestamp`
- [ ] `pop` está entre 0.0 e 1.0
- [ ] Silver tem ~160 linhas (não duplica após re-execução)
- [ ] Você entende o que `explode()` faz: 1 linha com array → N linhas

**Próximo: `05_build_gold_metrics.ipynb`**

---
*Workshop: Engenharia de Dados com Databricks & Delta Lake — OpenWeatherMap API*